FASE 1 

In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.read.format("delta").table("workspace.default.caso_practico_ventas")
df.display()

In [0]:
import pandas as pd

# 1. Convertir a Pandas
df_pd = df.toPandas()

# 2. Primeras 10 y últimas 5 filas
display(pd.concat([df_pd.head(10), df_pd.tail(5)]))

In [0]:
# 3. Verificar cantidad de registros y columnas

print("Número de registros:", df_pd.shape[0])
print("Número de columnas:", df_pd.shape[1])

In [0]:
# 4. Identificar tipos de datos
print("\n--- Tipos de datos ---")
print(df_pd.dtypes)
df_pd.info()
 
print("""
RESPUESTA: NO coinciden del todo con los tipos esperados:
 - fecha: está como texto y debería ser fecha (datetime).
 - cantidad y satisfaccion_cliente: están como decimal por los valores nulos y deberían ser enteros.
 - precio_unitario (decimal) y las demás columnas de texto sí coinciden.
""")

In [0]:
# 5. Cantidad de valores nulos por columna

nulos = pd.DataFrame({
    "Cantidad_nulos": df_pd.isnull().sum(),
    "Porcentaje_nulos": ((df_pd.isnull().sum() / len(df_pd)) * 100).round(2)
})
display(nulos.reset_index())

In [0]:
# Estadísticas solicitadas

estadisticas = pd.DataFrame({
    "Media": [
        df_pd["cantidad"].mean(),
        df_pd["precio_unitario"].mean()
    ],
    "Mediana": [
        df_pd["cantidad"].median(),
        df_pd["precio_unitario"].median()
    ],
    "Desviación estándar": [
        df_pd["cantidad"].std(),
        df_pd["precio_unitario"].std()
    ],
    "Mínimo": [
        df_pd["cantidad"].min(),
        df_pd["precio_unitario"].min()
    ],
    "Máximo": [
        df_pd["cantidad"].max(),
        df_pd["precio_unitario"].max()
    ]
}, index=["cantidad", "precio_unitario"])

estadisticas

FASE 2

In [0]:
df_pd.columns.tolist()

In [0]:
# ==============================
# FASE 2: LIMPIEZA DE DATOS
# ==============================

# 1. Eliminar registros duplicados
df_pd = df_pd.drop_duplicates()

# 2. Estandarizar método de pago
df_pd['metodo_pago'] = df_pd['metodo_pago'].replace(
    'Tarjeta Credito',
    'Tarjeta de Crédito'
)

# 3. Completar valores faltantes de cantidad con la mediana
df_pd['cantidad'] = df_pd['cantidad'].fillna(
    df_pd['cantidad'].median()
)

# 4. Completar valores faltantes de satisfacción con la moda
df_pd['satisfaccion_cliente'] = df_pd['satisfaccion_cliente'].fillna(
    df_pd['satisfaccion_cliente'].mode()[0]
)

# 5. Corregir el precio atípico 9999
# Se calcula el promedio por categoría de producto
promedio = df_pd.groupby(
    'categoria_producto'
)['precio_unitario'].transform('mean')

# Reemplazar únicamente el valor 9999
df_pd.loc[
    df_pd['precio_unitario'] == 9999,
    'precio_unitario'
] = promedio[
    df_pd['precio_unitario'] == 9999
]

# 6. Comprobación final
print("========== RESULTADO DE LA LIMPIEZA ==========")
print("Filas y columnas:", df_pd.shape)

print("\nValores nulos por columna:")
print(df_pd.isnull().sum())

print("\nCantidad de duplicados:")
print(df_pd.duplicated().sum())

print("\nPrimeras filas de la base limpia:")
display(df_pd.head())

FASE 3  

In [0]:
# ==============================
# FASE 3: TRANSFORMACIÓN E INGENIERÍA DE ATRIBUTOS
# ==============================

# 1. Crear la variable monto_total
df_pd['monto_total'] = df_pd['cantidad'] * df_pd['precio_unitario']


# 2. Convertir fecha a formato fecha
df_pd['fecha'] = pd.to_datetime(df_pd['fecha'])

# Extraer el día de la semana
df_pd['dia_semana'] = df_pd['fecha'].dt.day_name()

# Extraer el mes
df_pd['mes'] = df_pd['fecha'].dt.month


# 3. Crear rangos de monto_total
def clasificar_monto(monto):
    if monto < 100:
        return 'Bajo'
    elif monto <= 500:
        return 'Medio'
    else:
        return 'Alto'

df_pd['rango_monto'] = df_pd['monto_total'].apply(clasificar_monto)


# 4. Comprobar el resultado
print("========== FASE 3 ==========")
print("Nuevas columnas creadas:")
print("monto_total, dia_semana, mes y rango_monto")

display(df_pd.head())

print("\nCantidad de transacciones por rango:")
print(df_pd['rango_monto'].value_counts())

FASE 4 

In [0]:
# ==============================
# FASE 4: AGREGACIÓN Y RESUMEN
# ==============================

# 1. Group By:
# Total de ingresos y promedio de unidades vendidas por categoría
resumen_categoria = df_pd.groupby('categoria_producto').agg(
    ingresos_totales=('monto_total', 'sum'),
    promedio_unidades=('cantidad', 'mean')
).reset_index()

print("========== INGRESOS Y UNIDADES POR CATEGORÍA ==========")
display(resumen_categoria)


# 2. Tabla dinámica:
# Región en filas y categoría de producto en columnas
tabla_pivot = pd.pivot_table(
    df_pd,
    values='monto_total',
    index='region',
    columns='categoria_producto',
    aggfunc='sum',
    fill_value=0
)

print("========== TABLA DINÁMICA ==========")
display(tabla_pivot)


# 3. Nivel medio de satisfacción por método de pago
satisfaccion_pago = df_pd.groupby('metodo_pago').agg(
    satisfaccion_promedio=('satisfaccion_cliente', 'mean')
).reset_index()

print("========== SATISFACCIÓN POR MÉTODO DE PAGO ==========")
display(satisfaccion_pago)

FASE 5 

In [0]:
# ==============================
# FASE 5: SELECCIÓN, FILTRADO Y MUESTREO
# ==============================

# 1. Filtrado multicriterio
# Ventas de la región Norte con satisfacción >= 4
ventas_norte = df_pd[
    (df_pd['region'] == 'Norte') &
    (df_pd['satisfaccion_cliente'] >= 4)
]

print("========== FILTRO MULTICRITERIO ==========")
print("Cantidad de registros:", len(ventas_norte))
display(ventas_norte)


# 2. Muestreo estratificado por región
# Se toma el 10% de cada región
muestra_10 = (
    df_pd.groupby('region', group_keys=False)
    .apply(lambda x: x.sample(frac=0.10, random_state=42))
    .reset_index(drop=True)
)

print("========== MUESTRA ESTRATIFICADA ==========")
print("Cantidad de registros en la muestra:", len(muestra_10))
display(muestra_10)


# 3. Comprobar la proporción de regiones
print("========== PROPORCIÓN POR REGIÓN ==========")
print(muestra_10['region'].value_counts(normalize=True))